In [72]:
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import wandb
import h5py
import numpy as np
from tqdm import tqdm
import os
import torch.nn.functional as F
import pdb
from datetime import datetime
import json
from scipy.spatial.transform import Rotation as R
from diffusion_policy.model.common.rotation_transformer import RotationTransformer

In [167]:
rotation_transformer = RotationTransformer('axis_angle', 'rotation_6d')
temp = h5py.File('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images.hdf5')
idx=17
temp['data']['demo_1']['obs']['robot0_eef_pos'][idx]

current_pos = temp['data']['demo_0']['obs']['robot0_eef_pos'][idx]
future_pos = temp['data']['demo_0']['obs']['robot0_eef_pos'][(idx+1)]
current_rot = temp['data']['demo_0']['obs']['robot0_eef_quat'][idx]
future_rot = temp['data']['demo_0']['obs']['robot0_eef_quat'][(idx+1)]
rel_action = temp['data']['demo_0']['actions'][idx][:7]
print('current_pos',current_pos,
      '\n','rel_action',rel_action,
      '\n','future_pos',future_pos,
      '\n','current_rot',current_rot,
      '\n','future_rot',future_rot,
     )

# def transform(rel_action,current_pos):
raw_shape = rel_action.shape
d_rot = rel_action.shape[-1] - 4
rel_pos = rel_action[...,:3]
rot = rel_action[...,3:3+d_rot]
gripper = rel_action[...,[-1]]
abs_pos = rel_pos + current_pos
rot = rotation_transformer.forward(rot)
print(raw_shape,d_rot,rel_pos,rel_rot,gripper,'\n')
print(abs_pos,rot,gripper,'\n')

current_pos [ 1.3941607  -0.56687992  1.24907732] 
 rel_action [ 0.2        -0.41428571 -0.67142857 -0.         -0.          0.07142857
 -1.        ] 
 future_pos [ 1.40025251 -0.56495232  1.24092838] 
 current_rot [ 0.67966436  0.71947683  0.08875282 -0.11194818] 
 future_rot [ 0.67605216  0.72295945  0.08743009 -0.11242371]
(7,) 3 [ 0.2        -0.41428571 -0.67142857] [ 0.00981076 -0.05665955 -0.02698215] [-1.] 

[ 1.5941607  -0.98116563  0.57764875] [ 0.99745006 -0.07136785 -0.          0.07136785  0.99745006  0.        ] [-1.] 



In [137]:
temp = h5py.File('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images_val_classifier.hdf5')
temp

In [73]:
# 1. Define dataset
class RobotGoalDataset(Dataset):
    def __init__(self, actions, states, labels):
        self.actions = actions  # Shape: (N, 8, 7)
        self.states = states    # Shape: (N, 9)
        self.labels = labels    # Shape: (N,)
        print(self.actions.shape)
        print(self.states.shape)
        print(self.labels.shape)
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        action_flat = self.actions[idx].reshape(-1)  # Flatten to 56
        state = self.states[idx]
        input_vector = np.concatenate((action_flat, state))  # Final shape: (65,)
        return torch.tensor(input_vector, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.float32)

In [74]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=65):
        super(SimpleClassifier, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            # nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [75]:
def load_data(dataset_paths, experts):
    actions_list = []
    states_list = []
    labels = []
    for dataset_path, expert in zip(dataset_paths,experts):
        print(dataset_path, expert)
        hfile = h5py.File(dataset_path)
        
        for demo in tqdm(hfile['data']):
            demo_data = hfile['data'][demo]
            actions_data = demo_data['actions'][:]
            rewards = demo_data['rewards'][:]
            robot0_eef_pos = demo_data['obs']['robot0_eef_pos'][:]
            raw_obs = demo_data['raw_obs'][:]
        
            num_chunks = actions_data.shape[0] // 8
        
            for idx in range(num_chunks):
                a_vec = np.expand_dims(actions_data[idx*8:(idx+1)*8, :7], 0)
                if expert==True:
                    obs = np.concatenate([
                        robot0_eef_pos[idx*8+7],
                        raw_obs[idx, 3:6],
                        raw_obs[idx, 6:]
                    ], axis=0)
                else:
                    obs = np.concatenate([
                        robot0_eef_pos[idx*8+7],
                        raw_obs[idx, 14:17],
                        raw_obs[idx, 21:24]
                    ], axis=0)
                o_vec = np.expand_dims(obs, 0)
        
                actions_list.append(a_vec)
                states_list.append(o_vec)
                labels.append(np.max(rewards))
    
    # Stack once at the end
    actions = np.vstack(actions_list)
    states = np.vstack(states_list)
    labels = np.array(labels)
    hfile.close()
    return actions, states, labels

In [122]:
json.loads(temp['data'][demo].attrs['ep_meta'])['lang']

'pick the avocado from the sink and place it on the plate located on the counter'

In [130]:
temp= h5py.File('/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images_val.hdf5')
sets = set()
for demo in temp['data']:
    print(demo)
    sets.add(json.loads(temp['data'][demo].attrs['ep_meta'])['lang'].split('pick the ')[1].split(' from')[0])
print(sets)

demo_0
demo_1
demo_2
demo_3
demo_4
{'tangerine', 'avocado', 'corn', 'kiwi', 'bell pepper'}


In [114]:
dp1='/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1100-val_loss=0.037/for_classifier_training/PnPSinkToCounter_mg_val_kbpctk_firsthalf_4101143_midway_rollout_140_2530/datafile.hdf5'
# dp2='/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1100-val_loss=0.037/for_classifier_training/PnPSinkToCounter_mg_train_no_kbpctk_410115737_midway_rollout_140_2490/datafile.hdf5'
# dp3='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/mg/2024-05-04-22-14-34_and_2024-05-07-07-40-21/demo_gentex_im128_randcams_new_images_train_no_kbpckt_classifier.hdf5'
# dp4='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images_train_classifier.hdf5'
# dp5='/proj/vondrick3/sruthi/robots/robocasa/datasets/v0.1/single_stage/kitchen_pnp/PnPSinkToCounter/2024-04-26_2/demo_gentex_im128_randcams_new_images_val_classifier.hdf5'
dp6='/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1100-val_loss=0.037/apr15_classifier_training_data/PnPSinkToCounter_val_415132446_mr140_10samples/datafile.hdf5'
dpaths=[dp1,dp6]#,dp2,dp3,dp4,dp5]
are_experts=[False]#,False,True,True,True]
#load dataset
actions, states, labels = load_data(dpaths,are_experts)


/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1100-val_loss=0.037/for_classifier_training/PnPSinkToCounter_mg_val_kbpctk_firsthalf_4101143_midway_rollout_140_2530/datafile.hdf5 False


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2531/2531 [00:03<00:00, 819.99it/s]


In [115]:
# 4. Train-test split
train_actions, val_actions, train_states, val_states, train_labels, val_labels = train_test_split(
    actions, states, labels, test_size=0.2, random_state=42
)

# 5. Data loaders
train_dataset = RobotGoalDataset(train_actions, train_states, train_labels)
val_dataset = RobotGoalDataset(val_actions, val_states, val_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

num_pos = (train_labels == 1).sum()
num_neg = (train_labels == 0).sum()
print(num_neg, num_pos)

(107314, 8, 7)
(107314, 9)
(107314,)
(26829, 8, 7)
(26829, 9)
(26829,)
69798 37516


In [13]:
batch_size=4096
learning_rate=0.001
total_epochs=100

base_path = f'/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1100-val_loss=0.037/for_classifier_training'
now = datetime.now()  # ✅ Define it here
path_name = f'classifier_{now.strftime("%Y-%m-%d_%H-%M-%S")}'
os.makedirs(f'{base_path}/{path_name}', exist_ok=True)
run = wandb.init(project="robot-goal-classifier", name=path_name)

with open(f'{base_path}/{path_name}/deets.txt','w') as f:
    f.write('path_name: \n'+path_name+'\ndataset_paths: \n'+str(dpaths)+'\nwandburl: \n'+str(run.url) )

# 6. Training loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleClassifier().to(device)

# Calculate pos_weight: num_neg / num_pos
train_num_pos = (train_labels == 1).sum()
train_num_neg = (train_labels == 0).sum()
train_pos_weight = torch.tensor([train_num_neg / train_num_pos], dtype=torch.float32).to(device)

criterion = torch.nn.BCEWithLogitsLoss()#pos_weight=train_pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001)


wandb.config.update({
    "epochs": total_epochs,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "input_dim": 65,  # or 65 depending on your input
    "dataset_path": dpaths,
})


# Track best F1 and epoch
best_f1 = 0.0
best_epoch = 0

# 🧪 Evaluation
model.eval()
y_true = []
y_pred = []
with torch.no_grad():
    for inputs, targets in val_loader:
        outputs = model(inputs.to(device))
        probs = F.sigmoid(outputs)  # 🔁 Apply sigmoid manually
        predictions = (probs > 0.5).float().cpu().numpy()
        y_true.extend(targets.numpy())
        y_pred.extend(predictions)

report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

# 📊 Print classification report
print(classification_report(y_true, y_pred, digits=4, zero_division=0))


# 🔁 Training loop
for epoch in range(total_epochs):
    model.train()
    total_loss = 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Training Loss: {avg_train_loss:.4f}")

    # 🧪 Evaluation
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for inputs, targets in val_loader:
            outputs = model(inputs.to(device))
            probs = F.sigmoid(outputs)  # 🔁 Apply sigmoid manually
            predictions = (probs > 0.5).float().cpu().numpy()
            y_true.extend(targets.numpy())
            y_pred.extend(predictions)

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    # 📊 Print classification report
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    # 🪄 Log metrics to wandb
    val_macro_f1 = report["macro avg"]["f1-score"]

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_accuracy": report["accuracy"],
        "val_precision_class0": report["0.0"]["precision"],
        "val_recall_class0": report["0.0"]["recall"],
        "val_f1_class0": report["0.0"]["f1-score"],
        "val_precision_class1": report["1.0"]["precision"],
        "val_recall_class1": report["1.0"]["recall"],
        "val_f1_class1": report["1.0"]["f1-score"],
        "val_macro_f1": report["macro avg"]["f1-score"],
        "val_macro_precision": report["macro avg"]["precision"],
        "val_macro_recall": report["macro avg"]["recall"],
    })


    # Save best model checkpoint
    if val_macro_f1 > best_f1:
        best_f1 = val_macro_f1
        best_epoch = epoch + 1
        torch.save(model.state_dict(), f'{base_path}/{path_name}/best_path.pth')
        wandb.save('best_path.pth')
        print(f"✅ Saved new best model at epoch {best_epoch} with F1 = {best_f1:.4f}")
    torch.save(model.state_dict(), f'{base_path}/{path_name}/epoch_{epoch}.pth')


# 🛑 Finish the run
wandb.finish()

epoch,▁
train_loss,▁
val_accuracy,▁
val_f1_class0,▁
val_f1_class1,▁
val_macro_f1,▁
val_macro_precision,▁
val_macro_recall,▁
val_precision_class0,▁
val_precision_class1,▁
val_recall_class0,▁


              precision    recall  f1-score   support

         0.0     1.0000    0.0000    0.0001     37494
         1.0     0.4678    1.0000    0.6374     32950

    accuracy                         0.4678     70444
   macro avg     0.7339    0.5000    0.3187     70444
weighted avg     0.7510    0.4678    0.2982     70444

Epoch 1, Training Loss: 0.3783
              precision    recall  f1-score   support

         0.0     0.8757    0.8902    0.8829     37494
         1.0     0.8727    0.8562    0.8644     32950

    accuracy                         0.8743     70444
   macro avg     0.8742    0.8732    0.8736     70444
weighted avg     0.8743    0.8743    0.8742     70444

✅ Saved new best model at epoch 1 with F1 = 0.8736
Epoch 2, Training Loss: 0.2795
              precision    recall  f1-score   support

         0.0     0.9081    0.8741    0.8908     37494
         1.0     0.8626    0.8993    0.8806     32950

    accuracy                         0.8859     70444
   macro avg   

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇██
train_loss,█▆▅▄▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▃▃▆▆▆▆▇▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇█▇███▇▇█▇███
val_f1_class0,▁▁▃▄▄▅▅▆▅▆▆▇▆▇▇▇▆▇▇▇▇▇▇▇▇▇▇▇█▇█▇███████▇
val_f1_class1,▂▁▃▂▃▅▄▅▆▆▅▇▆▆▆▆▇▆▇▆▆▇▇▆▇▇▇█▇▅▇▇▇█▇▇▇▆▆█
val_macro_f1,▁▂▄▅▆▇▆▇▇▇▇▇▇▇▇█▇▇▇▇▇▇█▇█████▇▇██▇███▇██
val_macro_precision,▁▅▅▆▆▅▆▆▆▆▇▇▇▇▇▆▇▇███▇▇█▇▇▇██▇▇█▇████▇▇█
val_macro_recall,▁▂▄▅▅▆▆▆▇▆▇▇▇▇▆▇▇▇▇▇▇█▇▇██▇█▇▆▇████▇█▇██
val_precision_class0,▁▄▅▅▆▆▇█▇▇▇▇▇▇▇▇▇▇▇▇█▇▇▇▇█▇▇█▆█▇█▇█▇██▇█
val_precision_class1,▂▁▅▅▆▆▆▇▇▇█▅▇▇▆▆▇▆▆▇███▆▇▇▇▇█▆█▇▇▇██▇▇▇▇
val_recall_class0,▃▅▄▄▅▄▁▄▆▆▅▄▆▂▄██▇▇▆▇▇▅▆▇▄▄▇▆▇▄██▇█▅▇▅▆▆


In [158]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loaded_model = SimpleClassifier().to(device)
# loaded_model.load_state_dict(torch.load(f'{base_path}/{path_name}/best_path.pth'))
loaded_model.load_state_dict(torch.load('/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1100-val_loss=0.037/for_classifier_training/classifier_2025-04-14_15-46-23/best_path.pth'))
loaded_model.eval()

SimpleClassifier(
  (model): Sequential(
    (0): Linear(in_features=65, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [165]:
test_dataset_path=dp6#'/proj/vondrick3/sruthi/robots/diffusion_policy/data/outputs/2025.02.15/imageonly_11.32.40_usegroupnorm/checkpoints/epoch=1100-val_loss=0.037/PnPSinkToCounter_mg_val_kbpctk__4923942_midway_rollout_140_16/datafile.hdf5'
test_actions, test_states, test_labels = load_data([test_dataset_path],[True])

# Test Data loaders
test_dataset = RobotGoalDataset(test_actions, test_states, test_labels)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=True)

# 🧪 Test
loaded_model.eval()
y_true = []
y_pred = []
with torch.no_grad():
    for inputs, targets in test_loader:
        outputs = loaded_model(inputs.to(device))
        probs = F.sigmoid(outputs)  # 🔁 Apply sigmoid manually
        predictions = (outputs > 0.5).float().cpu().numpy()
        y_true.extend(targets.numpy())
        y_pred.extend(predictions)

report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

# 📊 Print classification report
print(classification_report(y_true, y_pred, digits=4, zero_division=0))


NameError: name 'dp6' is not defined